In [ ]:
---

## SMA 5 / SMA 100 Crossover vs Simple DCA vs Buy & Hold — Last 2 BTC Cycles

** Window: ** 3
rd
BTC
halving(May
11
2020) → present, spanning
two
complete
halving
cycles.

| Strategy | Entry / Exit
Logic |
| ---------- | ------------------- |
| ** SMA
5 / 100
Crossover ** | Long(all - in)
when
SMA - 5
crosses
above
SMA - 100;
exit
to
cash
on
cross - under |
| ** Simple
DCA ** | Invest
`$10, 000 ÷ n_trading_days
` each
day — same
total
capital as B & H |
| ** Buy & Hold ** | Full $10, 000
invested
on
the
first
day
of
Cycle
3, held
until
today |

** Starting
capital: ** $10, 000 · ** Fee: ** 0.1 % per
trade
** Cycle
boundary: ** vertical
dashed
line
at
April
19
2024(4
th
halving)
# ── Setup: cycle window + run all three strategies ───────────────────────────
CYCLE_3_START = '2020-05-11'  # 3rd BTC halving date
CYCLE_4_START = '2024-04-19'  # 4th BTC halving date (cycle boundary marker)
CYCLE_CAPITAL = 10_000
FEE = 0.001

cycle_price = price_s.loc[CYCLE_3_START:].copy()
n_days = len(cycle_price)
c4_mark = pd.Timestamp(CYCLE_4_START)

print(f'Window : {cycle_price.index[0].date()} → {cycle_price.index[-1].date()}  ({n_days} trading days)')
print(f'Cycle 3: {CYCLE_3_START} → {CYCLE_4_START}  '
      f'({(c4_mark - cycle_price.index[0]).days} days)')
print(f'Cycle 4: {CYCLE_4_START} → present  '
      f'({(cycle_price.index[-1] - c4_mark).days} days)')

# ── Strategy 1: SMA 5 / SMA 100 crossover ────────────────────────────────────
# Reuse MAs precomputed on the full price_s so the 100-day warmup
# is satisfied before the cycle window begins (no look-ahead bias).
fast_c = mas['SMA_5'].reindex(cycle_price.index)
slow_c = mas['SMA_100'].reindex(cycle_price.index)

p_arr = cycle_price.values
f_arr = fast_c.values
s_arr = slow_c.values

pos = (f_arr > s_arr).astype(int)
chg = np.diff(pos, prepend=pos[0])

cash_sma = float(CYCLE_CAPITAL);
btc_sma = 0.0
pv_sma_arr = np.empty(n_days);
n_trades = 0
buys = []  # list of (date, price) for buy signals
sells = []  # list of (date, price) for sell signals

for i in range(n_days):
    if chg[i] == 1 and cash_sma > 0:  # crossover up → buy all
        btc_sma = cash_sma * (1 - FEE) / p_arr[i]
        cash_sma = 0.0;
        n_trades += 1
        buys.append((cycle_price.index[i], p_arr[i]))
    elif chg[i] == -1 and btc_sma > 0:  # crossover down → sell all
        cash_sma = btc_sma * p_arr[i] * (1 - FEE)
        btc_sma = 0.0;
        n_trades += 1
        sells.append((cycle_price.index[i], p_arr[i]))
    pv_sma_arr[i] = cash_sma + btc_sma * p_arr[i]

pv_sma = pd.Series(pv_sma_arr, index=cycle_price.index)

# ── Strategy 2: Simple DCA ────────────────────────────────────────────────────
# Spreads the full $10,000 in equal daily instalments over the window,
# so total capital invested equals that of B&H.
daily_invest = CYCLE_CAPITAL / n_days
dca_cash = float(CYCLE_CAPITAL);
dca_btc = 0.0
pv_dca_arr = []

for p in p_arr:
    if dca_cash > 0.01:
        spend = min(daily_invest, dca_cash)
        dca_btc += spend * (1 - FEE) / p
        dca_cash -= spend
    pv_dca_arr.append(dca_cash + dca_btc * p)

pv_dca = pd.Series(pv_dca_arr, index=cycle_price.index)

# ── Strategy 3: Buy & Hold ────────────────────────────────────────────────────
bh_btc = CYCLE_CAPITAL * (1 - FEE) / p_arr[0]
pv_bh = pd.Series(bh_btc * p_arr, index=cycle_price.index)

# Colour palette (reused in chart + bar cells)
STRAT_COLORS = {'sma': '#58a6ff', 'dca': '#3fb950', 'bh': '#e6b800'}

print(f'\nSMA 5/100:  {n_trades} trades  ({len(buys)} buys · {len(sells)} sells)')
print(f'Simple DCA: ${daily_invest:.2f}/day for {n_days} days')
print()
print(f'Final portfolio values:')
print(f'  SMA 5/100  : ${pv_sma.iloc[-1]:>10,.0f}  '
      f'({(pv_sma.iloc[-1] / CYCLE_CAPITAL - 1) * 100:+.1f}%)')
print(f'  Simple DCA : ${pv_dca.iloc[-1]:>10,.0f}  '
      f'({(pv_dca.iloc[-1] / CYCLE_CAPITAL - 1) * 100:+.1f}%)')
print(f'  Buy & Hold : ${pv_bh.iloc[-1]:>10,.0f}  '
      f'({(pv_bh.iloc[-1] / CYCLE_CAPITAL - 1) * 100:+.1f}%)')
# ── 3-panel comparison chart ─────────────────────────────────────────────────
today_str_cycle = datetime.today().strftime('%Y-%m-%d')

fig, axes = plt.subplots(3, 1, figsize=(18, 20), facecolor='#0d1117',
                         gridspec_kw={'height_ratios': [2.5, 1.5, 3.0], 'hspace': 0.32})
HALVING_C = '#f0883e'

# ── Panel 1: Equity curves ───────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#161b22')
ax.plot(pv_sma.index, pv_sma.values, color=STRAT_COLORS['sma'], lw=1.8,
        label=f'SMA 5/100  →  ${pv_sma.iloc[-1]:,.0f}  '
              f'({(pv_sma.iloc[-1] / CYCLE_CAPITAL - 1) * 100:+.0f}%)')
ax.plot(pv_dca.index, pv_dca.values, color=STRAT_COLORS['dca'], lw=1.8,
        label=f'Simple DCA  →  ${pv_dca.iloc[-1]:,.0f}  '
              f'({(pv_dca.iloc[-1] / CYCLE_CAPITAL - 1) * 100:+.0f}%)')
ax.plot(pv_bh.index, pv_bh.values, color=STRAT_COLORS['bh'], lw=1.8,
        label=f'Buy & Hold  →  ${pv_bh.iloc[-1]:,.0f}  '
              f'({(pv_bh.iloc[-1] / CYCLE_CAPITAL - 1) * 100:+.0f}%)')
ax.axvline(c4_mark, color=HALVING_C, lw=1.2, ls='--', alpha=0.8, label='4th Halving (Apr 2024)')
ax.axhline(CYCLE_CAPITAL, color='white', lw=0.5, ls=':', alpha=0.25)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title(
    f'SMA 5/100 vs Simple DCA vs Buy & Hold  |  Last 2 BTC Cycles  |  {today_str_cycle}\n'
    f'$10,000 starting capital  ·  0.1% fee per trade',
    color='white', fontsize=13, fontweight='bold', pad=10
)
ax.legend(facecolor='#21262d', labelcolor='white', fontsize=10)
ax.grid(alpha=0.10)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#30363d')

# Add cycle labels
y_top = pv_bh.max() * 1.02
ax.text(pd.Timestamp('2021-06-01'), y_top, 'Cycle 3', color=HALVING_C,
        fontsize=9, alpha=0.7, ha='center')
ax.text(pd.Timestamp('2025-01-01'), y_top, 'Cycle 4', color=HALVING_C,
        fontsize=9, alpha=0.7, ha='center')

# ── Panel 2: Drawdown ────────────────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor('#161b22')
for pv, color, lbl in [(pv_sma, STRAT_COLORS['sma'], 'SMA 5/100'),
                       (pv_dca, STRAT_COLORS['dca'], 'Simple DCA'),
                       (pv_bh, STRAT_COLORS['bh'], 'Buy & Hold')]:
    dd = (pv - pv.cummax()) / pv.cummax() * 100
    ax.plot(dd.index, dd.values, color=color, lw=1.0, alpha=0.85, label=lbl)
    ax.fill_between(dd.index, dd.values, 0, color=color, alpha=0.05)
ax.axvline(c4_mark, color=HALVING_C, lw=1.2, ls='--', alpha=0.8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.set_title('Drawdown from Peak', color='white', fontsize=11, pad=6)
ax.legend(facecolor='#21262d', labelcolor='white', fontsize=9, ncol=3)
ax.grid(alpha=0.10)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#30363d')

# ── Panel 3: BTC price + SMA overlay + buy/sell trade markers ────────────────
ax = axes[2]
ax.set_facecolor('#161b22')
ax.plot(cycle_price.index, cycle_price.values,
        color='#444d56', lw=0.9, alpha=0.9, label='BTC Price')
ax.plot(fast_c.index, fast_c.values,
        color='#79c0ff', lw=1.2, alpha=0.85, label='SMA 5  (fast)')
ax.plot(slow_c.index, slow_c.values,
        color='#e6b800', lw=1.5, alpha=0.85, label='SMA 100  (slow)')

# Shade region where fast > slow (long position)
ax.fill_between(cycle_price.index, cycle_price.values, 0,
                where=(fast_c.values > slow_c.values),
                color='#3fb950', alpha=0.06, label='Long zone')

# Buy signals (green triangle up)
if buys:
    b_dates, b_prices = zip(*buys)
    ax.scatter(b_dates, b_prices, marker='^', color='#3fb950',
               s=90, zorder=5, label=f'Buy  ({len(buys)})')

# Sell signals (red triangle down)
if sells:
    s_dates, s_prices = zip(*sells)
    ax.scatter(s_dates, s_prices, marker='v', color='#f85149',
               s=90, zorder=5, label=f'Sell  ({len(sells)})')

ax.axvline(c4_mark, color=HALVING_C, lw=1.2, ls='--', alpha=0.8, label='4th Halving')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title(f'BTC Price  +  SMA 5 / SMA 100  +  Crossover Signals  |  {n_trades} total trades',
             color='white', fontsize=11, pad=6)
ax.legend(facecolor='#21262d', labelcolor='white', fontsize=9, ncol=4)
ax.grid(alpha=0.10)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#30363d')

plt.savefig('sma5_100_cycle_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


# ── Per-cycle metrics table + bar chart ─────────────────────────────────────

def compute_metrics(pv, label, start_cap=None):
    base = pv.iloc[0] if start_cap is None else start_cap
    final = pv.iloc[-1]
    total = (final / base - 1) * 100
    n_yrs = (pv.index[-1] - pv.index[0]).days / 365.25
    cagr = ((final / base) ** (1 / n_yrs) - 1) * 100 if n_yrs > 0.01 else 0.0
    dr = pv.pct_change().dropna()
    rf = (1.04 ** (1 / 365.25)) - 1
    std = dr.std()
    exc = dr - rf
    shrp = exc.mean() / std * np.sqrt(365.25) if std > 0 else 0.0
    down = dr[dr < rf]
    srt = exc.mean() / down.std() * np.sqrt(365.25) if (len(down) > 1 and down.std() > 0) else 0.0
    roll_max = pv.cummax()
    max_dd = ((pv - roll_max) / roll_max * 100).min()
    return {'Strategy': label,
            'Return (%)': f'{total:+.1f}', 'CAGR (%)': f'{cagr:.1f}',
            'Max DD (%)': f'{max_dd:.1f}', 'Sharpe': f'{shrp:.2f}',
            'Sortino': f'{srt:.2f}', 'Final ($)': f'${final:,.0f}'}


# ── Locate cycle 4 boundary in portfolio series ───────────────────────────────
c4_idx = cycle_price.index.searchsorted(c4_mark)  # first bar on/after 4th halving

# Full-window metrics (use CYCLE_CAPITAL as the base for all three so comparison is apples-to-apples)
rows_full = [
    {**compute_metrics(pv_sma, 'SMA 5/100 Crossover', CYCLE_CAPITAL), 'Trades': str(n_trades)},
    {**compute_metrics(pv_dca, 'Simple DCA', CYCLE_CAPITAL), 'Trades': f'{n_days} (daily)'},
    {**compute_metrics(pv_bh, 'Buy & Hold', CYCLE_CAPITAL), 'Trades': '1'},
]
df_full = pd.DataFrame(rows_full).set_index('Strategy')
col_order = ['Trades', 'Return (%)', 'CAGR (%)', 'Max DD (%)', 'Sharpe', 'Sortino', 'Final ($)']
df_full = df_full[col_order]

print('━' * 72)
print(f'  FULL WINDOW  ({cycle_price.index[0].date()} → {cycle_price.index[-1].date()})')
print('━' * 72)
display(df_full)

# Cycle-split metrics (return relative to value at start of each sub-period)
rows_c3 = [compute_metrics(pv_sma.iloc[:c4_idx], 'SMA 5/100'),
           compute_metrics(pv_dca.iloc[:c4_idx], 'Simple DCA'),
           compute_metrics(pv_bh.iloc[:c4_idx], 'Buy & Hold')]
rows_c4 = [compute_metrics(pv_sma.iloc[c4_idx:], 'SMA 5/100'),
           compute_metrics(pv_dca.iloc[c4_idx:], 'Simple DCA'),
           compute_metrics(pv_bh.iloc[c4_idx:], 'Buy & Hold')]

print(f'\n{"━" * 72}')
print(f'  CYCLE 3  ({cycle_price.index[0].date()} → {cycle_price.index[c4_idx].date()})')
print(f'{"━" * 72}')
display(pd.DataFrame(rows_c3).set_index('Strategy')[['Return (%)', 'CAGR (%)', 'Max DD (%)', 'Sharpe', 'Sortino']])

print(f'\n{"━" * 72}')
print(f'  CYCLE 4  ({cycle_price.index[c4_idx].date()} → {cycle_price.index[-1].date()})')
print(f'{"━" * 72}')
display(pd.DataFrame(rows_c4).set_index('Strategy')[['Return (%)', 'CAGR (%)', 'Max DD (%)', 'Sharpe', 'Sortino']])

# ── Per-cycle return bar chart ────────────────────────────────────────────────
strat_names = ['SMA 5/100', 'Simple DCA', 'Buy & Hold']
BAR_COLORS = [STRAT_COLORS['sma'], STRAT_COLORS['dca'], STRAT_COLORS['bh']]

ret_c3_vals = [float(r['Return (%)']) for r in rows_c3]
ret_c4_vals = [float(r['Return (%)']) for r in rows_c4]
ret_full_vals = [(pv.iloc[-1] / CYCLE_CAPITAL - 1) * 100
                 for pv in [pv_sma, pv_dca, pv_bh]]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#0d1117')
fig.suptitle('Strategy Returns by BTC Cycle', color='white', fontsize=13, fontweight='bold', y=1.03)

titles = [
    f'Cycle 3  ({cycle_price.index[0].year}–{cycle_price.index[c4_idx].year})',
    f'Cycle 4  ({cycle_price.index[c4_idx].year}–present)',
    'Full Window (Cycle 3 + 4)',
]
for ax, vals, title in zip(axes, [ret_c3_vals, ret_c4_vals, ret_full_vals], titles):
    ax.set_facecolor('#161b22')
    bars = ax.bar(strat_names, vals, color=BAR_COLORS, edgecolor='#30363d', linewidth=0.6, width=0.55)
    ypad = max(abs(v) for v in vals) * 0.04
    for bar, val in zip(bars, vals):
        ytext = bar.get_height() + ypad if val >= 0 else bar.get_height() - ypad * 2
        ax.text(bar.get_x() + bar.get_width() / 2, ytext,
                f'{val:+.0f}%', ha='center', va='bottom', color='white',
                fontsize=12, fontweight='bold')
    ax.axhline(0, color='white', lw=0.6, alpha=0.35)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold', pad=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.tick_params(colors='white', labelsize=9)
    plt.setp(ax.get_xticklabels(), color='white')
    ax.grid(axis='y', alpha=0.12)
    for spine in ax.spines.values():
        spine.set_color('#30363d')

plt.tight_layout()
plt.savefig('sma5_100_per_cycle_returns.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()